# 从 0 到 1：用 backtesting.py 做一个 A 股/ETF 回测小实验室

> 面向初学者。不要先背一堆术语。我们先让系统跑起来，再把每个零件拆开看。  
> 示例标的：`515450`（场内 ETF）和 `603993`（A 股股票）。  
> 工具：`AKShare` 下载数据，`backtesting.py` 做回测，Notebook 做交互式实验。

这份教程的核心心智模型只有一行：

```text
价格数据 → 因子/指标 → 买卖信号 → 回测成交 → 净值曲线 → 评价策略
```

我们会一步步实现它。

---

## 你会学到什么

- 怎么用 AKShare 下载 `515450` 和 `603993` 的日线数据；
- AKShare 是否支持分钟线，以及分钟线的限制；
- 什么是 OHLCV，为什么 backtesting.py 需要它；
- 什么是指标、因子、信号、策略；
- 5 个常见因子：动量、波动率、均线偏离、成交量放大、RSI；
- 4 个策略：买入持有、双均线、RSI 均值回归、多因子趋势过滤；
- 怎么看收益、回撤、夏普、交易次数；
- 怎么做简单参数优化；
- 怎么导出 HTML 报告。

> 本文是学习材料，不是投资建议。回测盈利不代表实盘盈利。

## 0. 环境准备

项目已使用 `uv` 管理依赖。推荐在项目根目录运行：

```bash
uv sync
uv run jupyter notebook
```

如果需要单独安装：

```bash
uv add backtesting akshare pandas numpy bokeh ipywidgets plotly yfinance notebook nbconvert ipykernel
```

下面先检查关键依赖是否存在。

In [ ]:
import importlib.util

for pkg in ["backtesting", "akshare", "pandas", "numpy", "bokeh", "ipywidgets", "plotly"]:
    print(f"{pkg:12s}", "OK" if importlib.util.find_spec(pkg) else "MISSING")

## 1. 先建立最小概念地图

量化回测听起来复杂，其实每个策略都逃不开这几个东西：

| 概念 | 白话解释 | 例子 |
|---|---|---|
| 数据 | 市场发生过什么 | 开盘、最高、最低、收盘、成交量 |
| 指标 | 从数据算出来的工具 | MA、RSI、波动率 |
| 因子 | 可能解释收益/风险的变量 | 动量、估值、成交量放大 |
| 信号 | 明确的买卖判断 | `MA20 > MA60` 买入 |
| 策略 | 信号 + 仓位 + 风控 + 成本 | 双均线策略 |
| 回测 | 拿历史数据模拟交易 | 2020-2026 跑一遍 |
| 净值 | 账户价值曲线 | 10 万变成 12 万 |
| 回撤 | 从高点跌下来多少 | 最惨亏了 20% |

记住：**因子不是策略**。因子只是一个变量，策略才是行动规则。

In [ ]:
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from backtesting import Backtest, Strategy
from backtesting.lib import crossover

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 160)

DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)

## 2. 数据：我们要下载什么？

我们使用两个标的：

| 代码 | 类型 | 说明 | AKShare 日线接口 | AKShare 分钟线接口 |
|---|---|---|---|---|
| `515450` | 场内 ETF | ETF，不是普通股票 | `fund_etf_hist_em` | `fund_etf_hist_min_em` |
| `603993` | A 股股票 | 洛阳钼业 | `stock_zh_a_hist` | `stock_zh_a_hist_min_em` |

### AKShare 支持分钟级数据吗？

支持，常见周期：

```text
1 / 5 / 15 / 30 / 60 分钟
```

但要注意：

- 1 分钟数据通常只返回近 5 个交易日；
- 1 分钟数据通常不复权；
- 分钟数据更适合短周期验证，不适合直接做多年长期回测；
- 初学者长期回测，先用日线。

## 3. 把 AKShare 数据统一成 backtesting.py 格式

backtesting.py 至少需要这些列：

```text
Open, High, Low, Close
```

我们再保留 `Volume` 和 AKShare 可能返回的扩展字段：

| 字段 | 含义 | 用途 |
|---|---|---|
| `Open` | 开盘价 | 回测必需 |
| `High` | 最高价 | 回测必需，止损止盈会用到 |
| `Low` | 最低价 | 回测必需，止损止盈会用到 |
| `Close` | 收盘价 | 算指标最常用 |
| `Volume` | 成交量 | 成交量因子、流动性过滤 |
| `Amount` | 成交额 | 活跃度、流动性 |
| `Amplitude` | 振幅 | 风险/波动 |
| `PctChange` | 涨跌幅 | 收益、动量 |
| `Turnover` | 换手率 | 活跃度 |

如果某个接口没有某列，就自动跳过。

In [ ]:
def standardize_akshare(raw, date_col):
    """把 AKShare 中文字段改成 backtesting.py 友好的英文列名。"""
    rename = {
        date_col: "Date",
        "开盘": "Open",
        "最高": "High",
        "最低": "Low",
        "收盘": "Close",
        "成交量": "Volume",
        "成交额": "Amount",
        "振幅": "Amplitude",
        "涨跌幅": "PctChange",
        "涨跌额": "Change",
        "换手率": "Turnover",
        "均价": "AvgPrice",
    }
    df = raw.rename(columns=rename).copy()
    df["Date"] = pd.to_datetime(df["Date"])
    df = df.set_index("Date").sort_index()

    keep = [
        "Open", "High", "Low", "Close", "Volume",
        "Amount", "Amplitude", "PctChange", "Change", "Turnover", "AvgPrice",
    ]
    keep = [c for c in keep if c in df.columns]

    for c in keep:
        df[c] = pd.to_numeric(df[c], errors="coerce")

    return df[keep].dropna(subset=["Open", "High", "Low", "Close"])

## 4. 下载日线数据

默认下载前复权日线：`adjust="qfq"`。

复权简单理解：

| 参数 | 含义 | 初学者怎么用 |
|---|---|---|
| `""` | 不复权 | 看真实历史成交价 |
| `"qfq"` | 前复权 | 看图、技术指标更顺 |
| `"hfq"` | 后复权 | 长期收益研究常用 |

这里用 `qfq`，因为图形连续，适合入门。

为了让教程可重复运行，我们把下载结果缓存到 `data/`。

In [ ]:
def load_515450_daily(start="20200101", end="20260101", adjust="qfq", use_cache=True):
    cache = DATA_DIR / f"515450_etf_daily_{start}_{end}_{adjust or 'none'}.csv"
    if use_cache and cache.exists():
        return pd.read_csv(cache, index_col="Date", parse_dates=True)

    import akshare as ak
    raw = ak.fund_etf_hist_em(
        symbol="515450",
        period="daily",
        start_date=start,
        end_date=end,
        adjust=adjust,
    )
    df = standardize_akshare(raw, "日期")
    df.to_csv(cache, index_label="Date")
    return df


def load_603993_daily(start="20200101", end="20260101", adjust="qfq", use_cache=True):
    cache = DATA_DIR / f"603993_stock_daily_{start}_{end}_{adjust or 'none'}.csv"
    if use_cache and cache.exists():
        return pd.read_csv(cache, index_col="Date", parse_dates=True)

    import akshare as ak
    raw = ak.stock_zh_a_hist(
        symbol="603993",
        period="daily",
        start_date=start,
        end_date=end,
        adjust=adjust,
        timeout=30,
    )
    df = standardize_akshare(raw, "日期")
    df.to_csv(cache, index_label="Date")
    return df

### 如果 AKShare 失败怎么办？

真实世界经常这样：网络失败、接口限流、数据源改字段。

为了让教程不断，我们准备一个假的 OHLCV 数据 fallback。它不能用于投资，只是保证代码能跑。

In [ ]:
def make_demo_ohlcv(seed=1, start="2020-01-01", periods=900):
    rng = np.random.default_rng(seed)
    dates = pd.bdate_range(start, periods=periods)
    ret = rng.normal(0.0003, 0.018, size=periods)
    close = 10 * np.exp(np.cumsum(ret))
    open_ = close * (1 + rng.normal(0, 0.004, size=periods))
    high = np.maximum(open_, close) * (1 + rng.random(periods) * 0.012)
    low = np.minimum(open_, close) * (1 - rng.random(periods) * 0.012)
    volume = rng.integers(100_000, 5_000_000, size=periods)
    return pd.DataFrame({
        "Open": open_,
        "High": high,
        "Low": low,
        "Close": close,
        "Volume": volume,
        "Amount": volume * close,
    }, index=dates)

In [ ]:
START = "20200101"
END = "20260101"

try:
    data_map = {
        "515450 ETF": load_515450_daily(START, END, adjust="qfq"),
        "603993 股票": load_603993_daily(START, END, adjust="qfq"),
    }
    print("AKShare 数据下载/读取成功")
except ModuleNotFoundError:
    print("未安装 akshare，使用演示数据。运行 `uv sync` 后可重新下载真实数据。")
    data_map = {
        "515450 ETF": make_demo_ohlcv(seed=515450),
        "603993 股票": make_demo_ohlcv(seed=603993),
    }
except Exception as e:
    print("AKShare 下载失败，使用演示数据。错误：", repr(e))
    data_map = {
        "515450 ETF": make_demo_ohlcv(seed=515450),
        "603993 股票": make_demo_ohlcv(seed=603993),
    }

for name, df in data_map.items():
    print(f"{name}: {df.index.min().date()} → {df.index.max().date()}, shape={df.shape}")
    display(df.tail())

## 5. 分钟线下载函数：先放着，不默认运行

分钟线可以下载，但本教程主线用日线。原因很简单：

```text
日线适合先学概念，分钟线会引入更多噪声、数据限制和成交细节。
```

下面两个函数给你以后用。

In [ ]:
def load_515450_min(start="2026-01-01 09:30:00", end="2026-08-28 15:00:00", period="5", adjust=""):
    import akshare as ak
    raw = ak.fund_etf_hist_min_em(
        symbol="515450",
        start_date=start,
        end_date=end,
        period=period,
        adjust=adjust,
    )
    return standardize_akshare(raw, "时间")


def load_603993_min(start="2026-01-01 09:30:00", end="2026-08-28 15:00:00", period="5", adjust=""):
    import akshare as ak
    raw = ak.stock_zh_a_hist_min_em(
        symbol="603993",
        start_date=start,
        end_date=end,
        period=period,
        adjust=adjust,
    )
    return standardize_akshare(raw, "时间")

# 示例，不默认运行：
# df_5m = load_603993_min(period="5")
# df_5m.tail()

## 6. 五个入门因子

我们先只用价格和成交量。够了，别一上来搞复杂。

| 因子 | 代码名 | 直觉 |
|---|---|---|
| 20 日动量 | `mom_20` | 最近涨得多，可能还强 |
| 20 日波动率 | `vol_20` | 最近波动大，风险更高 |
| 20 日均线偏离 | `ma_gap_20` | 价格离均线有多远 |
| 成交量放大 | `volume_ratio_20` | 今天是否明显放量 |
| 14 日 RSI | `rsi_14` | 是否短期超买/超卖 |

因子本身不下单。我们后面把因子变成信号。

In [ ]:
def SMA(values, n):
    return pd.Series(values).rolling(n).mean()


def RSI(values, n=14):
    values = pd.Series(values)
    diff = values.diff()
    gain = diff.clip(lower=0)
    loss = -diff.clip(upper=0)
    rs = gain.ewm(alpha=1 / n, adjust=False).mean() / loss.ewm(alpha=1 / n, adjust=False).mean()
    return 100 - 100 / (1 + rs)


def add_factors(df):
    out = df.copy()
    ret = out["Close"].pct_change()

    out["mom_20"] = out["Close"].pct_change(20)
    out["vol_20"] = ret.rolling(20).std() * np.sqrt(252)
    out["ma_gap_20"] = out["Close"] / out["Close"].rolling(20).mean() - 1
    out["volume_ratio_20"] = out["Volume"] / out["Volume"].rolling(20).mean()
    out["rsi_14"] = RSI(out["Close"], 14).values

    if "Amount" in out.columns:
        out["amount_ratio_20"] = out["Amount"] / out["Amount"].rolling(20).mean()
    if "Turnover" in out.columns:
        out["turnover_ma_20"] = out["Turnover"].rolling(20).mean()

    return out

factor_map = {name: add_factors(df) for name, df in data_map.items()}

for name, df in factor_map.items():
    print(name)
    cols = [c for c in ["Close", "Volume", "Amount", "Turnover", "mom_20", "vol_20", "ma_gap_20", "volume_ratio_20", "rsi_14"] if c in df.columns]
    display(df[cols].tail())

### 看一眼因子图

图不是为了“找圣杯”。只是让你形成直觉：价格涨跌、动量、波动、RSI 是如何一起变化的。

In [ ]:
def plot_factors(name):
    df = factor_map[name]
    try:
        import plotly.graph_objects as go
        from plotly.subplots import make_subplots

        fig = make_subplots(
            rows=4,
            cols=1,
            shared_xaxes=True,
            vertical_spacing=0.03,
            subplot_titles=("Close", "Momentum / MA Gap", "Volatility", "RSI"),
        )
        fig.add_trace(go.Scatter(x=df.index, y=df["Close"], name="Close"), row=1, col=1)
        fig.add_trace(go.Scatter(x=df.index, y=df["mom_20"], name="mom_20"), row=2, col=1)
        fig.add_trace(go.Scatter(x=df.index, y=df["ma_gap_20"], name="ma_gap_20"), row=2, col=1)
        fig.add_trace(go.Scatter(x=df.index, y=df["vol_20"], name="vol_20"), row=3, col=1)
        fig.add_trace(go.Scatter(x=df.index, y=df["rsi_14"], name="rsi_14"), row=4, col=1)
        fig.add_hline(y=70, line_dash="dot", row=4, col=1)
        fig.add_hline(y=30, line_dash="dot", row=4, col=1)
        fig.update_layout(height=850, title=f"{name}: 因子直觉图")
        fig.show()
    except ImportError:
        df[["Close", "mom_20", "ma_gap_20", "vol_20", "rsi_14"]].plot(subplots=True, figsize=(12, 8))

plot_factors("515450 ETF")

## 7. backtesting.py 的策略写法

策略就是一个类。你只需要关心两个函数：

| 函数 | 什么时候运行 | 做什么 |
|---|---|---|
| `init()` | 回测开始前运行一次 | 预计算指标 |
| `next()` | 每根 K 线运行一次 | 判断买卖 |

注意一个非常重要的点：

```text
默认情况下，今天产生信号，下一根 K 线成交。
```

这能避免很多“偷看未来”的错误。

## 8. 策略一：买入并持有

这是基准。任何策略都应该和它比。

规则：

```text
第一次有机会就买入，然后一直持有。
```

In [ ]:
class BuyAndHold(Strategy):
    def init(self):
        pass

    def next(self):
        if not self.position:
            self.buy()

## 9. 策略二：双均线趋势

最经典的趋势策略。

规则：

```text
快均线上穿慢均线：买入
快均线下穿慢均线：卖出
```

直觉：如果短期价格强于长期价格，趋势可能向上。

In [ ]:
class SmaCrossStrategy(Strategy):
    fast = 20
    slow = 60

    def init(self):
        self.ma_fast = self.I(SMA, self.data.Close, self.fast)
        self.ma_slow = self.I(SMA, self.data.Close, self.slow)

    def next(self):
        if crossover(self.ma_fast, self.ma_slow):
            self.position.close()
            self.buy()
        elif crossover(self.ma_slow, self.ma_fast):
            self.position.close()

## 10. 策略三：RSI 均值回归

均值回归的想法：跌多了可能反弹，涨多了可能休息。

规则：

```text
RSI < 30：短期超卖，买入
RSI > 55：反弹后，卖出
```

风险：如果是单边下跌，低 RSI 可能只是“越跌越低”。

In [ ]:
class RsiMeanReversionStrategy(Strategy):
    rsi_window = 14
    buy_level = 30
    sell_level = 55

    def init(self):
        self.rsi = self.I(RSI, self.data.Close, self.rsi_window)

    def next(self):
        if not self.position and self.rsi[-1] < self.buy_level:
            self.buy()
        elif self.position and self.rsi[-1] > self.sell_level:
            self.position.close()

## 11. 策略四：多因子趋势过滤

我们把多个因子组合起来。

买入条件：

```text
20 日动量 > 3%
价格 > 60 日均线
20 日年化波动率 < 45%
成交量不太差：Volume / Volume_MA20 > 0.7
RSI < 80，避免太热
```

卖出条件：

```text
动量转负，或价格跌破 60 日均线，或 RSI > 85
```

这不是最优策略，只是展示“因子如何变成规则”。

In [ ]:
class MultiFactorTrendStrategy(Strategy):
    mom_window = 20
    ma_window = 60
    vol_window = 20
    min_mom = 0.03
    max_vol = 0.45
    min_volume_ratio = 0.7
    max_entry_rsi = 80
    max_exit_rsi = 85

    def init(self):
        self.mom = self.I(lambda x: pd.Series(x).pct_change(self.mom_window), self.data.Close, name="mom_20")
        self.ma = self.I(SMA, self.data.Close, self.ma_window, name="ma_60")
        self.vol = self.I(
            lambda x: pd.Series(x).pct_change().rolling(self.vol_window).std() * np.sqrt(252),
            self.data.Close,
            name="vol_20",
            plot=False,
        )
        self.volume_ratio = self.I(
            lambda x: pd.Series(x) / pd.Series(x).rolling(20).mean(),
            self.data.Volume,
            name="volume_ratio_20",
            plot=False,
        )
        self.rsi = self.I(RSI, self.data.Close, 14, name="rsi_14")

    def next(self):
        price = self.data.Close[-1]
        if np.isnan(self.mom[-1]) or np.isnan(self.ma[-1]) or np.isnan(self.vol[-1]):
            return

        enter = (
            self.mom[-1] > self.min_mom and
            price > self.ma[-1] and
            self.vol[-1] < self.max_vol and
            self.volume_ratio[-1] > self.min_volume_ratio and
            self.rsi[-1] < self.max_entry_rsi
        )
        exit_ = (
            self.mom[-1] < 0 or
            price < self.ma[-1] or
            self.rsi[-1] > self.max_exit_rsi
        )

        if not self.position and enter:
            self.buy()
        elif self.position and exit_:
            self.position.close()

## 12. 统一运行回测

为了少写重复代码，我们封装两个函数：

- `run_backtest()`：跑一次回测；
- `summarize_stats()`：抽取关键指标。

这里成本先用简化佣金 `commission=0.0003`。真实 A 股还要考虑：卖出印花税、最小 100 股、T+1、停牌、涨跌停、滑点。

> 教程先保持简单。真实交易制度以后再补，不要第一步就把自己劝退。

In [ ]:
METRICS = [
    "Start", "End", "Duration",
    "Exposure Time [%]",
    "Equity Final [$]",
    "Return [%]",
    "Buy & Hold Return [%]",
    "Max. Drawdown [%]",
    "Sharpe Ratio",
    "Calmar Ratio",
    "# Trades",
    "Win Rate [%]",
    "Profit Factor",
    "SQN",
]

STRATEGIES = {
    "买入并持有": (BuyAndHold, {}),
    "双均线趋势": (SmaCrossStrategy, {"fast": 20, "slow": 60}),
    "RSI 均值回归": (RsiMeanReversionStrategy, {"rsi_window": 14, "buy_level": 30, "sell_level": 55}),
    "多因子趋势过滤": (MultiFactorTrendStrategy, {}),
}


def run_backtest(df, strategy_cls, params=None, cash=100_000, commission=0.0003):
    bt = Backtest(
        df,
        strategy_cls,
        cash=cash,
        commission=commission,
        exclusive_orders=True,
        finalize_trades=True,
    )
    stats = bt.run(**(params or {}))
    return bt, stats


def summarize_stats(symbol_name, strategy_name, stats):
    row = {"标的": symbol_name, "策略": strategy_name}
    for m in METRICS:
        row[m] = stats.get(m)
    return row

In [ ]:
rows = []
backtest_cache = {}

for symbol_name, df in data_map.items():
    for strategy_name, (strategy_cls, params) in STRATEGIES.items():
        bt, stats = run_backtest(df, strategy_cls, params)
        backtest_cache[(symbol_name, strategy_name)] = (bt, stats)
        rows.append(summarize_stats(symbol_name, strategy_name, stats))

summary = pd.DataFrame(rows)
summary[["标的", "策略", "Return [%]", "Buy & Hold Return [%]", "Max. Drawdown [%]", "Sharpe Ratio", "# Trades", "Win Rate [%]"]]

## 13. 怎么读回测结果？

不要只看收益。建议按这个顺序看：

1. 有没有跑赢买入持有；
2. 最大回撤能不能接受；
3. 交易次数是否太少；
4. 胜率和盈亏比是否匹配；
5. 成本后是否还有效；
6. 换参数后是否仍然有效。

下面写一个很小的解释函数。

In [ ]:
def explain_stats(stats):
    ret = stats.get("Return [%]", np.nan)
    bh = stats.get("Buy & Hold Return [%]", np.nan)
    dd = stats.get("Max. Drawdown [%]", np.nan)
    trades = stats.get("# Trades", 0)
    sharpe = stats.get("Sharpe Ratio", np.nan)

    lines = []
    if pd.notna(ret) and pd.notna(bh):
        lines.append(f"策略收益 {ret:.2f}%，买入持有 {bh:.2f}%，{'跑赢' if ret > bh else '跑输'}基准。")
    if pd.notna(dd):
        lines.append(f"最大回撤 {dd:.2f}%。这是历史上从高点到低点最难受的一段。")
    if trades < 10:
        lines.append(f"交易次数 {trades} 次，样本偏少，别太相信统计指标。")
    else:
        lines.append(f"交易次数 {trades} 次，比单笔运气更有参考价值。")
    if pd.notna(sharpe):
        lines.append(f"夏普比率 {sharpe:.2f}。它衡量收益和波动的性价比，但不能单独看。")
    return "\n".join(lines)

_, demo_stats = backtest_cache[("515450 ETF", "双均线趋势")]
print(explain_stats(demo_stats))

## 14. 交互式图表

`bt.plot()` 会生成 backtesting.py 的交互式图表，包括：

- K 线；
- 指标；
- 买卖区间；
- 净值曲线；
- 成交量；
- 交易盈亏。

下面默认看 `515450 ETF` 的双均线策略。

In [ ]:
bt, stats = backtest_cache[("515450 ETF", "双均线趋势")]
display(stats[METRICS])
bt.plot(open_browser=False)

## 15. 交易明细和净值曲线

`stats` 里有两个很有用的表：

| 字段 | 内容 |
|---|---|
| `_trades` | 每笔交易 |
| `_equity_curve` | 每天账户权益和回撤 |

它们是以后做自定义报告的原材料。

In [ ]:
display(stats["_trades"].tail())
display(stats["_equity_curve"].tail())

## 16. 参数优化：双均线

现在我们用顺序网格搜索参数。它与小规模 `Backtest.optimize(method="grid")` 的目标相同，但在 macOS/Jupyter 中更稳定，不会触发多进程共享内存错误。

但请记住：优化不是为了找到神奇数字，而是看策略是否稳定。

```text
好现象：一片参数区域都还可以
坏现象：只有一个孤零零的参数点特别好
```

In [ ]:
from itertools import product

df = data_map["515450 ETF"]
bt_opt = Backtest(
    df,
    SmaCrossStrategy,
    cash=100_000,
    commission=0.0003,
    exclusive_orders=True,
    finalize_trades=True,
)

# Backtest.optimize() 默认会使用多进程共享内存；在部分 macOS + Jupyter
# 环境会触发 resource_tracker KeyError。参数组合很少，顺序搜索更稳定。
results = {}
for fast, slow in product(range(5, 31, 5), range(20, 121, 20)):
    if fast >= slow:
        continue
    stats = bt_opt.run(fast=fast, slow=slow)
    results[(fast, slow)] = stats["Sharpe Ratio"]

heatmap = pd.Series(
    results,
    index=pd.MultiIndex.from_tuples(results, names=["fast", "slow"]),
    name="Sharpe Ratio",
)
valid = heatmap.dropna()
if valid.empty:
    raise ValueError("所有参数组合的 Sharpe Ratio 都为空，请检查数据和交易次数。")

best_fast, best_slow = valid.idxmax()
best_stats = bt_opt.run(fast=best_fast, slow=best_slow)

# 最小自检：选出的参数确实对应热力图最大值。
assert np.isclose(best_stats["Sharpe Ratio"], valid.max())

print(f"最优参数：fast={best_fast}, slow={best_slow}")
display(best_stats[METRICS])
heatmap_df = heatmap.unstack()
display(heatmap_df.style.background_gradient(cmap="RdYlGn", axis=None))

## 17. Notebook 交互式调参

如果安装了 `ipywidgets`，下面可以用控件改参数。适合新手形成直觉。

In [ ]:
try:
    import ipywidgets as widgets
    from IPython.display import display, clear_output

    def interactive_run(symbol_name, strategy_name, fast, slow, rsi_buy, rsi_sell, commission):
        clear_output(wait=True)
        df = data_map[symbol_name]

        if strategy_name == "双均线趋势":
            if fast >= slow:
                print("要求 fast < slow")
                return
            strategy_cls, params = SmaCrossStrategy, {"fast": fast, "slow": slow}
        elif strategy_name == "RSI 均值回归":
            strategy_cls, params = RsiMeanReversionStrategy, {"buy_level": rsi_buy, "sell_level": rsi_sell}
        elif strategy_name == "多因子趋势过滤":
            strategy_cls, params = MultiFactorTrendStrategy, {}
        else:
            strategy_cls, params = BuyAndHold, {}

        _, s = run_backtest(df, strategy_cls, params, commission=commission)
        print(explain_stats(s))
        display(s[METRICS])
        display(s["_trades"].tail(10))

    display(widgets.interactive(
        interactive_run,
        symbol_name=widgets.Dropdown(options=list(data_map.keys()), description="标的"),
        strategy_name=widgets.Dropdown(options=list(STRATEGIES.keys()), description="策略"),
        fast=widgets.IntSlider(value=20, min=5, max=80, step=5, description="fast"),
        slow=widgets.IntSlider(value=60, min=20, max=180, step=10, description="slow"),
        rsi_buy=widgets.IntSlider(value=30, min=10, max=50, step=5, description="RSI买"),
        rsi_sell=widgets.IntSlider(value=55, min=45, max=90, step=5, description="RSI卖"),
        commission=widgets.FloatSlider(value=0.0003, min=0, max=0.005, step=0.0001, readout_format=".4f", description="佣金"),
    ))
except ImportError:
    print("未安装 ipywidgets。需要交互控件时运行：uv add ipywidgets")

## 18. 单标的、多标的、组合回测，不是一回事

本教程做的是：**同一套策略分别跑 515450 和 603993**。

| 类型 | 含义 | 本教程是否覆盖 |
|---|---|---|
| 单标的回测 | 一个标的、一套策略、一条资金曲线 | 覆盖 |
| 多标的批量回测 | 多个标的分别跑，最后对比 | 覆盖 |
| 组合回测 | 多个标的共享一个账户，同时持仓 | 没完整覆盖 |
| 因子选股回测 | 按因子排序选 Top N，定期调仓 | 只讲概念 |

backtesting.py 很适合单标的教学。真正做股票池组合、因子分组，多数时候 `vectorbt` 会更顺手。

## 19. A 股真实交易里还缺什么？

本教程故意保持简单，所以没有完整模拟：

| 真实细节 | 为什么重要 |
|---|---|
| T+1 | 今天买入，通常不能今天卖出 |
| 涨跌停 | 信号出现了也不一定能成交 |
| 停牌 | 没有价格，无法交易 |
| 100 股整数倍 | 仓位不能无限精确 |
| 印花税 | 股票卖出成本更高，ETF 通常不同 |
| 滑点 | 信号价不等于成交价 |
| 流动性 | 成交量太小，买不进去也卖不出来 |

这不是小问题。但新手第一步先理解回测流程，再补交易制度。

## 20. 未来函数和过拟合：新手两大坑

### 未来函数

未来函数就是用了当时不知道的信息。

错误例子：

```text
今天收盘后才知道收盘价，却假设今天收盘前已经买入。
```

backtesting.py 默认“下一根 K 线成交”，能帮你少踩一部分坑。

### 过拟合

过拟合就是策略记住了历史噪声。

危险信号：

- 参数很多；
- 优化后收益突然夸张；
- 换一段时间就失效；
- 只有一个参数组合赚钱，附近都不赚钱；
- 交易次数很少但指标很好。

最懒的防过拟合方法：策略简单一点，参数少一点，看样本外。

## 21. AI 写策略：先生成配置，不要直接执行任意代码

如果以后接 AI，最安全的第一版是：

```text
自然语言 → JSON 参数 → 调用已写好的策略模板
```

例如：

```json
{
  "symbol": "603993 股票",
  "strategy": "双均线趋势",
  "params": {"fast": 20, "slow": 60},
  "commission": 0.0003
}
```

不要一开始就让 AI 生成任意 Python 并执行。那是安全坑。

In [ ]:
def run_from_config(config):
    df = data_map[config["symbol"]]
    strategy_cls, default_params = STRATEGIES[config["strategy"]]
    params = {**default_params, **config.get("params", {})}
    _, s = run_backtest(df, strategy_cls, params, commission=config.get("commission", 0.0003))
    return s

config = {
    "symbol": "603993 股票",
    "strategy": "双均线趋势",
    "params": {"fast": 20, "slow": 60},
    "commission": 0.0003,
}

s = run_from_config(config)
print(explain_stats(s))

## 22. 导出 HTML

导出当前 Notebook：

```bash
uv run jupyter nbconvert --to html notebooks/backtesting_py_交互式实战.ipynb
```

先执行全部代码，再导出：

```bash
uv run jupyter nbconvert \
  --to html \
  --execute \
  --ExecutePreprocessor.timeout=600 \
  notebooks/backtesting_py_交互式实战.ipynb
```

生成文件：

```text
notebooks/backtesting_py_交互式实战.html
```

## 23. 下一步

如果你想把它变成一个小产品，最短路径：

1. 用 Streamlit 做页面；
2. 左边选标的、策略、参数；
3. 中间显示指标表、图表、交易明细；
4. 策略先只支持模板；
5. AI 先只生成 JSON 配置；
6. 真正股票池组合回测，再考虑 vectorbt。

不要自研回测引擎。先把现成工具用明白。

## 参考资料

backtesting.py：

- 官网：https://kernc.github.io/backtesting.py/
- API：https://kernc.github.io/backtesting.py/doc/backtesting/backtesting.html
- Quick Start：https://kernc.github.io/backtesting.py/doc/examples/Quick%20Start%20User%20Guide.html
- Multiple Time Frames：https://kernc.github.io/backtesting.py/doc/examples/Multiple%20Time%20Frames.html
- Parameter Heatmap & Optimization：https://kernc.github.io/backtesting.py/doc/examples/Parameter%20Heatmap%20%26%20Optimization.html

AKShare：

- 股票数据：https://akshare.akfamily.xyz/data/stock/stock.html
- 基金 / ETF 数据：https://akshare.akfamily.xyz/data/fund/fund_public.html